In [111]:
#pip install hyperopt jupyter matplotlib pandas requests reservoirpy scikit-learn seaborn brevitas fxpmath

# Classification with Reservoir Computing

Reservoir Computing (RC) is well suited to both regression and classification tasks. In the following notebook, you will experiment with a simple example of classification task.

In [1]:
from collections import defaultdict
import matplotlib.pyplot as plt
import numpy as np
import reservoirpy as rpy

from reservoirpy.nodes import Reservoir, Ridge, Input
from reservoirpy.datasets import japanese_vowels
from reservoirpy import set_seed, verbosity
from reservoirpy.observables import nrmse, rsquare
from sklearn.metrics import recall_score, f1_score

from sklearn.metrics import accuracy_score

set_seed(42)
verbosity(0)

0

In [37]:
import numpy as np
from aeon.datasets import load_classification
from tensorflow.keras.utils import to_categorical
import reservoirpy as rp
import reservoirpy.nodes as rpn
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Set up reservoirpy parameters and configurations
rp.verbosity(0)  # No verbosity for clean output
rp.set_seed(42)  # Set random seed for reproducibility

# Load the training and testing data for the 'ECG200' dataset
X_train, y_train = load_classification('ECGFiveDays', split='train')
X_test, y_test = load_classification('ECGFiveDays', split='test')

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

# Check the unique labels in y
unique_labels = np.unique(y_train)
print(f"Unique labels in y_train: {unique_labels}")

# Convert string labels to integers using LabelEncoder
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_test)  # Use the same encoder to transform test labels

# One-hot encode the labels for the classification task
y_train_cat = to_categorical(y_train, num_classes=len(unique_labels))
y_test_cat = to_categorical(y_test, num_classes=len(unique_labels))

# Reshape the data to match the expected input shape (n_samples, n_features)
X_train_reshaped = X_train.reshape(X_train.shape[0], X_train.shape[2])  # Convert to 2D shape (samples, features)
X_test_reshaped = X_test.reshape(X_test.shape[0], X_test.shape[2])  # Convert to 2D shape (samples, features)

# Set up a range of N values to test
N_values = [250]  # Test N from 100 to 500
accuracies = []  # List to store the accuracy for each N

# Loop through different values of N
for N in N_values:
    print(f"\nTraining and evaluating model with N = {N} reservoir units...")
    
    # Build the Reservoir Computing Model
    reservoir = rpn.Reservoir(
        units=N,                # Number of reservoir neurons
        input_dim=X_train_reshaped.shape[1],  # Each time step in the time series is a vector of length n_features
        sr=0.6,                 # Spectral radius (for the echo state property)
        input_connectivity=0.9, # Spectral radius for input weights
        rc_connectivity=0.25,    # Reservoir internal connectivity
        input_scaling=1,     # Scaling for the input weights
        lr=0.6,                 # Leaking rate (1.0 for non-leaky units)
    )

    # Create a Ridge regression readout node with the regularization parameter
    readout = rpn.Ridge(ridge=1e-4)

    # Build a pipeline: the reservoir feeds its output into the readout
    esn_model = reservoir >> readout

    # Train the Model
    esn_model.fit(X_train_reshaped, y_train_cat)

    # Evaluate the Model
    y_pred = esn_model.run(X_test_reshaped)

    # Convert the network outputs (continuous predictions) to class labels
    y_pred_labels = np.argmax(y_pred, axis=1)

    # Since y_test is already one-hot encoded, we need to convert it to the same format for comparison
    y_test_labels = np.argmax(y_test_cat, axis=1)

    # Compute the accuracy of the model
    accuracy = np.mean(y_pred_labels == y_test_labels)  # Correct accuracy calculation
    accuracies.append(accuracy)  # Store the accuracy for this N

    print(f"Test accuracy for N={N}: {accuracy * 100:.2f}%")

# Plot the results
# plt.figure(figsize=(10, 6))
# plt.plot(N_values, accuracies, marker='o', linestyle='-', color='b')
# plt.title('Accuracy vs. Reservoir Size (N)', fontsize=14)
# plt.xlabel('Number of Reservoir Units (N)', fontsize=12)
# plt.ylabel('Test Accuracy (%)', fontsize=12)
# plt.grid(True)
# plt.show()

# Print the best result
best_N = N_values[np.argmax(accuracies)]
best_accuracy = max(accuracies)
print(f"Best accuracy: {best_accuracy * 100:.5f}% with N = {best_N}")


X_train shape: (23, 1, 136), y_train shape: (23,)
X_test shape: (861, 1, 136), y_test shape: (861,)
Unique labels in y_train: ['1' '2']

Training and evaluating model with N = 250 reservoir units...
Test accuracy for N=250: 85.71%
Best accuracy: 85.71429% with N = 250


In [20]:
X= np.append(X_train_reshaped, X_test_reshaped, axis=0)

In [21]:
esn_model = esn_model.fit(X_train_reshaped, y_train_cat)
 #states of each time stamp over test set
reservoir_states_test = reservoir.run(X_test)  # Shape: (time_steps, num_neurons).
reservoir_states_test.shape
np.save('reservoir_states_test', reservoir_states_test)
output_test = readout.run(reservoir_states_test)  # Shape: (time_steps, output_dim)
output_test.shape
output_test = np.argmax(output_test, axis=1)
np.save('output_test', output_test)

In [24]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from brevitas.nn import QuantHardTanh, QuantIdentity, QuantLinear
from brevitas.quant import Int8ActPerTensorFloat, Int8WeightPerTensorFloat
from reservoirpy.datasets import mackey_glass
from reservoirpy.nodes import Reservoir, Ridge
from reservoirpy.observables import rmse, rsquare
from reservoirpy import Node
import reservoirpy as rpy
rpy.verbosity(0)
rpy.set_seed(42)
# pruned_res_size = len(ind)
#N = 100
# # Parameters
# N = pruned_res_size #reservoir size
bit_width = 8 # quantization bit width
#integer, out , scale,  quantizer, layer , ...
# input

from brevitas.nn import QuantIdentity
import math
def extract_Qinput(input, num_bits=8):
  quant_identity = QuantIdentity(return_quant_tensor=True,bit_width=num_bits)
  float_input = torch.tensor(input,dtype=torch.float64)
  quant_input = quant_identity(float_input)
  out = quant_input.int().detach().numpy()

  scale = quant_input.scale.detach().numpy()
  zero_point = quant_input.zero_point.numpy()
  print(f"Input int quantized:\n {out} \n")
  print(f"Input scale: {scale}")
  print(f"Input zero_point: {zero_point}")
  return out, scale, zero_point
from brevitas.nn import QuantLinear
from brevitas.quant import Int8ActPerTensorFloat,Int8WeightPerTensorFloat
import math


def quantize_output_layer(weights, num_bits=8):
  quant_linear = QuantLinear(2, 4,weight_bit_width=num_bits, weight_quant=Int8WeightPerTensorFloat, requires_grad=False)
  quant_linear.weight.data = torch.tensor(weights,dtype=torch.float64)

  out = quant_linear.quant_weight().int().detach().numpy()
  scale = quant_linear.quant_weight().scale.detach().numpy()
  # out = quant_linear.quant_weight().value.data.numpy()


  #out = out * 0

  return out,scale

int_x,x_scale,x_zero_point = extract_Qinput(X,num_bits=bit_width)
# np.savetxt('input_data.csv', int_x[5002:], fmt='%s', delimiter=',')
print('Extract quantized integer Win and Scale and Zero_point')
int_Win,scale_Win,zero_point_Win = extract_Qinput(esn_model.nodes[0].Win.todense(),num_bits=bit_width)
print('Extract quantized integer Wr and Scale and Zero_point')
int_Wr,scale_Wr,zero_point_Wr = extract_Qinput(esn_model.nodes[0].W.todense(),num_bits=bit_width)
int_bias, scale_bias, zero_point_bias = extract_Qinput(esn_model.nodes[0].bias.todense(), num_bits=bit_width)


input_scale = (scale_Win * x_scale)
threshold_scale = 0.0078125
reservoir_scale = (scale_Wr * threshold_scale)

def compute_integer_thresholds(scale):

    a = -1  # Lower bound of Hard Tanh
    b = 1   # Upper bound of Hard Tanh
    a_scaled = np.int32(a / scale)
    b_scaled = np.int32(b / scale)
    return a_scaled, b_scaled

def piecewise_linear_hard_tanh_integer(x, a_scaled, b_scaled):

    # Apply saturation logic
    x = np.where(x < a_scaled, a_scaled, x)
    x = np.where(x > b_scaled, b_scaled, x)
    x=x + b_scaled
    return (x / 256).astype(np.int32)

a_scaled, b_scaled = compute_integer_thresholds(input_scale)
c_scaled, d_scaled = compute_integer_thresholds(reservoir_scale)

from reservoirpy import Node
def forward2(node: Node, x: np.ndarray) -> np.ndarray:
    state = node.state()  # get current node state  # Current reservoir state as integer
    state_int = state.astype(np.int64)
    W_res = node.Wr
    W_in = node.Win
    bias_res = node.Bias
    state_int = state_int.reshape(1,N)

    r = state_int @ W_res.astype(np.int32)
    s = x @ W_in.astype(np.int32).T
    s = s.reshape(1,N)
    # Precompute thresholds and scale
    # a_scaled, b_scaled = compute_integer_thresholds(scale_Wr)
    bias_res = bias_res.reshape(1, N)



    # Apply integer PLA to reservoir and input contributions
    out_r = piecewise_linear_hard_tanh_integer(s,a_scaled, b_scaled)
    out_s = piecewise_linear_hard_tanh_integer(r,c_scaled, d_scaled)

    return out_r + out_s 

def initialize(node: Node, x: np.ndarray = None, y: np.ndarray = None):
    """This function receives a data point x at runtime and uses it to
    infer input and output dimensions.
    """
    if x is not None:
        node.set_input_dim(x.shape[1])
        node.set_output_dim(N)
        node.set_param("Wr", int_Wr)
        node.set_param("Win", int_Win)
        node.set_param("Bias", int_bias)

class CustomNode(Node):
    def __init__(self, const2=-1, name=None):
        super().__init__(
            forward=forward2,
            initializer=initialize,

            params={"const1": None, "Wr":None, "Win":None, "Bias":None},
            hypers={"const2": const2},
            name=name,
        )

node = CustomNode(const2=-1, name="custom_node3")
# readout_pruned.ridge = 1e-12

esn_model_PTQ = node >> readout

Quantized_States = node.run(int_x[:23].astype(np.float64))

Quantized_States = Quantized_States * threshold_scale
readout.fit(Quantized_States.astype(np.float64), y_train_cat)
# esn_model_PTQ = node >> readout_pruned

# esn_model_PTQ = esn_model_PTQ.fit(int_x[:5000].astype(np.float64), int_x[1:5001].astype(np.float64), warmup=1000)

Quantized_States = node.run(int_x[23:].astype(np.float64))
Quantized_States = Quantized_States * threshold_scale
Y_pred_PTQ_origin = readout.run(Quantized_States.astype(np.float64))

Y_pred_PTQ_origin = np.argmax(Y_pred_PTQ_origin, axis=1)

accuracy_PTQ_origin = np.mean(Y_pred_PTQ_origin == y_test_labels)
recall_val_PTQ_origin = recall_score(y_test_labels, Y_pred_PTQ_origin, average='macro')
f1_val_PTQ_origin = f1_score(y_test_labels, Y_pred_PTQ_origin, average='macro')
print(f"Test Recall_PTQ_origin for N={N}: {recall_val_PTQ_origin:.4f}")
print(f"Test F1-score_PTQ_origin for N={N}: {f1_val_PTQ_origin:.4f}")
print(f"Test accuracy Quantized Original: {accuracy_PTQ_origin * 100:.5f}%")




Input int quantized:
 [[ 0  1  0 ... -2  0  0]
 [ 1  1  1 ... -7 -7 -7]
 [34  7  6 ... -5 -5 -6]
 ...
 [ 2  2  2 ... -7 -7 -7]
 [ 1  1  1 ... -8 -7 -8]
 [ 0  1 -1 ... -4 -5 -4]] 

Input scale: 0.05548296015625
Input zero_point: 0.0
Extract quantized integer Win and Scale and Zero_point
Input int quantized:
 [[-128 -128  127 ... -128 -128 -128]
 [ 127 -128 -128 ... -128 -128  127]
 [-128  127 -128 ... -128 -128  127]
 ...
 [-128  127  127 ... -128 -128 -128]
 [-128    0 -128 ... -128  127  127]
 [-128  127  127 ... -128  127 -128]] 

Input scale: 0.0078125
Input zero_point: 0.0
Extract quantized integer Wr and Scale and Zero_point
Input int quantized:
 [[ -6   0   0 ...  -2   0  -1]
 [  0 -27   0 ...   0   0   0]
 [  0   0   0 ...   0   0  15]
 ...
 [  0  28   0 ... -63   0  -1]
 [  0  10   0 ...   0   0   0]
 [ 37   0 -32 ...   2   0   0]] 

Input scale: 0.002298181127078754
Input zero_point: 0.0
Input int quantized:
 [[-128]
 [   0]
 [-128]
 [ 127]
 [ 127]
 [-128]
 [ 127]
 [   0]
 [-1

In [25]:
from reservoirpy.nodes import Reservoir
rpy.verbosity(0)  # no need to be too verbose here
rpy.set_seed(42)  # make everything reproducible!

def create_new_reservoir(model, pruned_W, pruned_W_in, pruned_bias):


    # Create a new reservoir with the pruned configuration
    pruned_reservoir = Reservoir(
        len(pruned_bias.todense()),
        lr=model.nodes[0].lr,
        sr=model.nodes[0].sr,
        input_connectivity=model.nodes[0].input_connectivity,
        # rc_connectivity=model.nodes[0].rc_connectivity,
        input_scaling=(model.nodes[0].input_scaling),
        input_dim=model.nodes[0].input_dim
    )

    # Assign the pruned weights to the new reservoir
    pruned_reservoir.W = pruned_W
    pruned_reservoir.Win = pruned_W_in
    # pruned_reservoir.bias = pruned_bias

    return pruned_reservoir

In [44]:
# ///////////////////////////////////////HSIC_Lasso\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\
def HSIC_Prune():
    import numpy as np
    from pyHSICLasso import HSICLasso
    from sklearn.linear_model import LinearRegression
    from sklearn.metrics import mean_squared_error
    from sklearn.model_selection import train_test_split
    n, d = 4999, N  # n samples, d features
    np.random.seed(0)
    M = reservoir_states_test # Input matrix -> here should be the outcome of neurons
    y = output_test # Output of the ESN network

    # Your target RMSE
    target_rmse = 0.1  # Adjust this value based on your requirement
    tolerance = 0.1   # Acceptable tolerance around the target RMSE

    M_train, M_test, y_train, y_test = train_test_split(M, y, test_size=0.5, random_state=42)

    # Iterate over the number of features
    best_features = None
    best_rmse_diff = float(0.5)  # Difference between current RMSE and target RMSE
    int_list_temp=[]
    for num_features in range(1, d):
        print(num_features)
        hsic_lasso = HSICLasso()

        # Run HSIC Lasso to select a specific number of features
        hsic_lasso.input(M_train, y_train.reshape(-1))
        hsic_lasso.regression(num_feat=num_features)
        # hsic_lasso.plot()
        hsic_lasso.dump()

        selected_indices = hsic_lasso.get_features()
        selected_indices_temp = set(map(int, selected_indices))
        for i in range(len(selected_indices)):
            selected_indices_temp.update(hsic_lasso.get_index_neighbors(feat_index=i,num_neighbors=5))
            # selected_indices.append(hsic_lasso.get_index_neighbors(feat_index=i,num_neighbors=5))
        selected_indices = sorted(list(selected_indices_temp))
        if not selected_indices:  # Skip iteration if no features were selected
            continue
        # print(f"selected_indices: {selected_indices}")
        model = LinearRegression()
        int_list = list(map(int, selected_indices))
        if (int_list==int_list_temp):
            break
        print(f"int_list: {int_list}")
        ind = [x - 1 for x in int_list]
        print(f"ind: {ind}")
        int_list_temp=int_list
        model.fit(M_train[:, ind], y_train.reshape(-1))

        # Calculate RMSE on the test set
        y_pred = model.predict(M_test[:, ind])
        rmse_hsic = np.sqrt(mean_squared_error(y_test, y_pred))
        print(f"Found features with num_features={num_features}, RMSE={rmse_hsic} within tolerance of target RMSE.")
    new_size = len(ind)
    print("New reservoir size",len(ind))
    prune_count = new_size

    keep_neurons = ind
    pruned_W = esn_model.nodes[0].W[keep_neurons][:, keep_neurons]
    pruned_W_in = esn_model.nodes[0].Win[keep_neurons, :]
    pruned_W_out = esn_model.nodes[1].Wout[keep_neurons, :]
    pruned_bias = esn_model.nodes[0].bias[keep_neurons,:]

    reservoir_pruned = create_new_reservoir(esn_model, pruned_W, pruned_W_in, pruned_bias)

    readout_pruned = Ridge(ridge=readout.ridge)
    # readout_pruned.bias=readout.bias
    esn_model_pruned = reservoir_pruned >> readout_pruned

    readout_pruned.Wout=pruned_W_out
    # readout_pruned.bias = readout.bias
    esn_model_pruned=esn_model_pruned.fit(X_train_reshaped, y_train_cat)
    Y_pred_pruned = esn_model_pruned.run(X_test_reshaped)


    reservoir_pruned.W = pruned_W
    reservoir_pruned.Win = pruned_W_in
    reservoir_pruned.bias = pruned_bias
    esn_model_pruned=esn_model_pruned.fit(X_train_reshaped, y_train_cat)
    Y_pred_pruned = esn_model_pruned.run(X_test_reshaped)
    
    # Main ERROR
    Y_pred_pruned = np.argmax(Y_pred_pruned, axis=1)
    
    from sklearn.metrics import mean_squared_error # Import mean_squared_error from sklearn.metrics
    import math # Import math library

    print(reservoir_pruned.units)

    accuracy_pruned = np.mean(Y_pred_pruned == y_test_labels)
    print(f"Test accuracy HSIC Pruning: {accuracy_pruned * 100:.5f}%")
    return accuracy_pruned,prune_count,num_features,esn_model_pruned

In [45]:
from reservoirpy.observables import rmse, rsquare, mse, nrmse
from reservoirpy.nodes import Reservoir, Ridge
from sklearn.metrics import recall_score, f1_score
rpy.verbosity(0)  # no need to be too verbose here
rpy.set_seed(42)  # make everything reproducible!

# reservoir_sizes = [20,103,150, 190,250]  # Increase the reservoir size incrementally
reservoir_sizes = [250]
# Performance tracking
results = []

# Loop through different values of N
for N in reservoir_sizes:
    print(f"\nTraining and evaluating model with N = {N} reservoir units...")
    
    # Build the Reservoir Computing Model
    reservoir = Reservoir(
        units=N,                # Number of reservoir neurons
        input_dim=X_train_reshaped.shape[1],  # Each time step in the time series is a vector of length n_features
        sr=0.6,                 # Spectral radius (for the echo state property)
        input_connectivity=0.9, # Spectral radius for input weights
        rc_connectivity=0.25,    # Reservoir internal connectivity
        input_scaling=1,     # Scaling for the input weights
        lr=0.6,                 # Leaking rate (1.0 for non-leaky units)
    )

    # Create a Ridge regression readout node with the regularization parameter
    readout = Ridge(ridge=1e-4)
    # Build a pipeline: the reservoir feeds its output into the readout
    esn_model = reservoir >> readout

    # Train the Model
    esn_model.fit(X_train_reshaped, y_train_cat)

    # 'run' propagates the test data through the reservoir and readout.
    reservoir_states_test = reservoir.run(X_test_reshaped)  # Shape: (time_steps, num_neurons).
    output_test = readout.run(reservoir_states_test)  # Shape: (time_steps, output_dim)
    output_test = np.argmax(output_test, axis=1)

    states = reservoir_states_test
    Y_pred = output_test

    # Convert the network outputs (one-hot style predictions) to class labels.
    y_pred_labels = Y_pred

    # Since y_test is already one-hot encoded, we need to convert it to the same format for comparison
    y_test_labels = np.argmax(y_test_cat, axis=1)
    output_test = y_test_labels
    # Compute the accuracy of the model
    accuracy_val = np.mean(y_pred_labels == y_test_labels)  # Correct accuracy calculation
    print(f"Test accuracy for N={N}: {accuracy_val * 100:.2f}%")
    recall_val = recall_score(y_test_labels, y_pred_labels, average='macro')
    f1_val = f1_score(y_test_labels, y_pred_labels, average='macro')

    print(f"Test accuracy for N={N}: {accuracy_val * 100:.2f}%")
    print(f"Test Recall for N={N}: {recall_val:.4f}")
    print(f"Test F1-score for N={N}: {f1_val:.4f}")
    print(f"Test accuracy for N={N}: {accuracy_val * 100:.2f}%")

    # HSIC
    accuracy_pruned_HSIC,prune_count,num_features=HSIC_Prune()
    print("number of nuerons after pruning",prune_count)
    # Spearman
    accuracy_pruned_spearman=spearman_Prune(N - prune_count,num_features)
    # PCA
    accuracy_pruned_pca=PCA_Prune(N - prune_count,num_features)
    # lasso
    accuracy_pruned_lasso=lasso_Prune(N - prune_count,num_features)


    # Save results for comparison
    results.append({
        "N": N,
        "accuracy": accuracy_val,
        "recall": recall_val,
        "f1_score": f1_val,
        "pruned_size":prune_count,
        "accuracy_HSIC": accuracy_pruned_HSIC,
        "accuracy_Spearman": accuracy_pruned_spearman,
        "accuracy_PCA": accuracy_pruned_pca,
        "accuracy_Lasso": accuracy_pruned_lasso
    })
    print(f"Reservoir Size {N}: accuracy_original={accuracy_val:.4f}")
    print(f"Reservoir Size {N}: recall_original={recall_val:.4f}")
    print(f"Reservoir Size {N}: f1_score_original={f1_val:.4f}")
    print(f"Reservoir Size {N}: accuracy_HSIC={accuracy_pruned_HSIC:.4f}")
    print(f"Reservoir Size {N}: accuracy_Spearman={accuracy_pruned_spearman:.4f}")
    print(f"Reservoir Size {N}: accuracy_PCA={accuracy_pruned_pca:.4f}")
    print(f"Reservoir Size {N}: accuracy_Lasso={accuracy_pruned_lasso:.4f}")
for result in results:
    print(result)


Training and evaluating model with N = 250 reservoir units...
Test accuracy for N=250: 85.71%
Test accuracy for N=250: 85.71%
Test Recall for N=250: 0.8572
Test F1-score for N=250: 0.8571
Test accuracy for N=250: 85.71%
1
Block HSIC Lasso B = 20.
M set to 3.
Using Gaussian kernel for the features, Gaussian kernel for the outcomes.
============================================== HSICLasso : Result ==================================================
| Order | Feature      | Score | Top-5 Related Feature (Relatedness Score)                                          |
| 1     | 1            | nan | 174          (0.546), 38           (0.540), 12           (0.526), 25           (0.525), 31           (0.435)|
int_list: [1, 11, 24, 30, 37, 173]
ind: [0, 10, 23, 29, 36, 172]
Found features with num_features=1, RMSE=0.47744083484498656 within tolerance of target RMSE.
2
Block HSIC Lasso B = 20.
M set to 3.
Using Gaussian kernel for the features, Gaussian kernel for the outcomes.


/home/atousa/.local/lib/python3.10/site-packages/pyHSICLasso/api.py:107: RuntimeWarning: B 20 must be an exact divisor of the number of samples 430. Number of blocks 21.5 will be approximated to 21.
  warnings.warn(msg, RuntimeWarning)
/home/atousa/.local/lib/python3.10/site-packages/pyHSICLasso/kernel_tools.py:46: RuntimeWarning: overflow encountered in exp
  K = np.exp(-dist_2 / (2 * np.power(sigma, 2)))
/home/atousa/.local/lib/python3.10/site-packages/pyHSICLasso/api.py:176: RuntimeWarning: invalid value encountered in float_scalars
  self.beta[self.A[i]
/home/atousa/.local/lib/python3.10/site-packages/pyHSICLasso/api.py:107: RuntimeWarning: B 20 must be an exact divisor of the number of samples 430. Number of blocks 21.5 will be approximated to 21.
  warnings.warn(msg, RuntimeWarning)
/home/atousa/.local/lib/python3.10/site-packages/pyHSICLasso/kernel_tools.py:46: RuntimeWarning: overflow encountered in exp
  K = np.exp(-dist_2 / (2 * np.power(sigma, 2)))


============================================== HSICLasso : Result ==================================================
| Order | Feature      | Score | Top-5 Related Feature (Relatedness Score)                                          |
| 1     | 1            | nan | 174          (0.546), 38           (0.540), 12           (0.526), 25           (0.525), 31           (0.435)|
New reservoir size 6


/home/atousa/.local/lib/python3.10/site-packages/pyHSICLasso/api.py:176: RuntimeWarning: invalid value encountered in float_scalars
  self.beta[self.A[i]


6
Test accuracy HSIC Pruning: 60.27875%


ValueError: too many values to unpack (expected 3)

In [35]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from brevitas.nn import QuantHardTanh, QuantIdentity, QuantLinear
from brevitas.quant import Int8ActPerTensorFloat, Int8WeightPerTensorFloat
from reservoirpy.datasets import mackey_glass
from reservoirpy.nodes import Reservoir, Ridge
from reservoirpy.observables import rmse, rsquare
from reservoirpy import Node
import reservoirpy as rpy
rpy.verbosity(0)
rpy.set_seed(42)
pruned_res_size = esn_model_pruned.nodes[0].units

# Parameters
N = pruned_res_size #reservoir size
bit_width = 8 # quantization bit width
#integer, out , scale,  quantizer, layer , ...
# input

from brevitas.nn import QuantIdentity
import math
def extract_Qinput(input, num_bits=8):
  quant_identity = QuantIdentity(return_quant_tensor=True,bit_width=num_bits)
  float_input = torch.tensor(input,dtype=torch.float64)
  quant_input = quant_identity(float_input)
  out = quant_input.int().detach().numpy()

  scale = quant_input.scale.detach().numpy()
  zero_point = quant_input.zero_point.numpy()
  print(f"Input int quantized:\n {out} \n")
  print(f"Input scale: {scale}")
  print(f"Input zero_point: {zero_point}")
  return out, scale, zero_point
from brevitas.nn import QuantLinear
from brevitas.quant import Int8ActPerTensorFloat,Int8WeightPerTensorFloat
import math


def quantize_output_layer(weights, num_bits=8):
  quant_linear = QuantLinear(2, 4,weight_bit_width=num_bits, weight_quant=Int8WeightPerTensorFloat, requires_grad=False)
  quant_linear.weight.data = torch.tensor(weights,dtype=torch.float64)

  out = quant_linear.quant_weight().int().detach().numpy()
  scale = quant_linear.quant_weight().scale.detach().numpy()
  # out = quant_linear.quant_weight().value.data.numpy()


  #out = out * 0

  return out,scale

int_x,x_scale,x_zero_point = extract_Qinput(X,num_bits=bit_width)
# np.savetxt('input_data.csv', int_x[5002:], fmt='%s', delimiter=',')
print('Extract quantized integer Win and Scale and Zero_point')
int_Win,scale_Win,zero_point_Win = extract_Qinput(esn_model_pruned.nodes[0].Win.todense(),num_bits=bit_width)
print('Extract quantized integer Wr and Scale and Zero_point')
int_Wr,scale_Wr,zero_point_Wr = extract_Qinput(esn_model_pruned.nodes[0].W.todense(),num_bits=bit_width)
int_bias, scale_bias, zero_point_bias = extract_Qinput(esn_model_pruned.nodes[0].bias.todense(), num_bits=bit_width)


input_scale = (scale_Win * x_scale)
threshold_scale = 0.0078125
reservoir_scale = (scale_Wr * threshold_scale)

def compute_integer_thresholds(scale):

    a = -1  # Lower bound of Hard Tanh
    b = 1   # Upper bound of Hard Tanh
    a_scaled = np.int32(a / scale)
    b_scaled = np.int32(b / scale)
    return a_scaled, b_scaled

def piecewise_linear_hard_tanh_integer(x, a_scaled, b_scaled):

    # Apply saturation logic
    x = np.where(x < a_scaled, a_scaled, x)
    x = np.where(x > b_scaled, b_scaled, x)
    x=x + b_scaled
    return (x / 256).astype(np.int32)

a_scaled, b_scaled = compute_integer_thresholds(input_scale)
c_scaled, d_scaled = compute_integer_thresholds(reservoir_scale)

from reservoirpy import Node
def forward2(node: Node, x: np.ndarray) -> np.ndarray:
    state = node.state()  # get current node state  # Current reservoir state as integer
    state_int = state.astype(np.int64)
    W_res = node.Wr
    W_in = node.Win
    bias_res = node.Bias
    state_int = state_int.reshape(1,N)

    r = state_int @ W_res.astype(np.int32)
    s = x @ W_in.astype(np.int32).T
    s = s.reshape(1,N)
    # Precompute thresholds and scale
    # a_scaled, b_scaled = compute_integer_thresholds(scale_Wr)
    bias_res = bias_res.reshape(1, N)



    # Apply integer PLA to reservoir and input contributions
    out_r = piecewise_linear_hard_tanh_integer(s,a_scaled, b_scaled)
    out_s = piecewise_linear_hard_tanh_integer(r,c_scaled, d_scaled)

    return out_r + out_s 

def initialize(node: Node, x: np.ndarray = None, y: np.ndarray = None):
    """This function receives a data point x at runtime and uses it to
    infer input and output dimensions.
    """
    if x is not None:
        node.set_input_dim(x.shape[1])
        node.set_output_dim(N)
        node.set_param("Wr", int_Wr)
        node.set_param("Win", int_Win)
        node.set_param("Bias", int_bias)

class CustomNode(Node):
    def __init__(self, const2=-1, name=None):
        super().__init__(
            forward=forward2,
            initializer=initialize,

            params={"const1": None, "Wr":None, "Win":None, "Bias":None},
            hypers={"const2": const2},
            name=name,
        )

node = CustomNode(const2=-1, name="custom_node3")
# readout_pruned.ridge = 1e-12

esn_model_PTQ = node >> esn_model_pruned.nodes[1]

Quantized_States = node.run(int_x[:23].astype(np.float64))

Quantized_States = Quantized_States * threshold_scale
esn_model_pruned.nodes[1].fit(Quantized_States.astype(np.float64), y_train_cat)
# esn_model_PTQ = node >> readout_pruned

# esn_model_PTQ = esn_model_PTQ.fit(int_x[:5000].astype(np.float64), int_x[1:5001].astype(np.float64), warmup=1000)

Quantized_States = node.run(int_x[23:].astype(np.float64))
Quantized_States = Quantized_States * threshold_scale
Y_pred_PTQ = esn_model_pruned.nodes[1].run(Quantized_States.astype(np.float64))
# min_pred = np.min(Y_pred_PTQ)
# max_pred = np.max(Y_pred_PTQ)
# Y_pred_PTQ_normalized = 2 * (Y_pred_PTQ - min_pred) / (max_pred - min_pred) - 1

# Wout_quantized, scale_out = quantize_output_layer(readout.Wout,num_bits=bit_width)
# readout.Wout = Wout_quantized

# a=threshold_scale * scale_out

# Y_pred_PTQ = esn_model_PTQ.run(int_x[5001:-1])
# Y_pred_PTQ = Y_pred_PTQ *a
Y_pred_PTQ = np.argmax(Y_pred_PTQ, axis=1)
accuracy_PTQ = np.mean(Y_pred_PTQ == y_test_labels)
recall_val_PTQ = recall_score(y_test_labels, Y_pred_PTQ, average='macro')
f1_val_PTQ = f1_score(y_test_labels, Y_pred_PTQ, average='macro')
print(f"Test Recall_PTQ for N={N}: {recall_val_PTQ:.4f}")
print(f"Test F1-score_PTQ for N={N}: {f1_val_PTQ:.4f}")
print(f"Test accuracy PTQ: {accuracy_PTQ * 100:.5f}%")


Input int quantized:
 [[ 0  1  0 ... -2  0  0]
 [ 1  1  1 ... -7 -7 -7]
 [34  7  6 ... -5 -5 -6]
 ...
 [ 2  2  2 ... -7 -7 -7]
 [ 1  1  1 ... -8 -7 -8]
 [ 0  1 -1 ... -4 -5 -4]] 

Input scale: 0.05548296015625
Input zero_point: 0.0
Extract quantized integer Win and Scale and Zero_point
Input int quantized:
 [[-128 -128  127  127  127 -128 -128 -128  127 -128  127  127 -128 -128
   127  127 -128  127  127 -128  127  127 -128    0  127 -128    0 -128
   127  127  127 -128  127  127 -128  127  127  127 -128 -128  127  127
  -128  127  127  127  127  127  127 -128 -128    0  127 -128 -128  127
   127  127 -128 -128 -128  127 -128 -128  127  127 -128  127 -128  127
  -128  127  127 -128  127 -128    0  127 -128 -128  127 -128    0  127
  -128 -128 -128 -128    0  127 -128 -128 -128  127 -128 -128 -128  127
   127 -128  127 -128    0 -128  127  127 -128 -128 -128 -128  127 -128
   127 -128  127  127  127 -128  127  127 -128 -128 -128  127  127 -128
  -128  127  127 -128 -128  127 -128 -128 -